# 🥭 Training YOLO11 untuk Deteksi Kematangan Mangga (Mango Ripeness Detection)

Notebook ini disiapkan khusus untuk melatih model **YOLO11 Nano (`yolo11n.pt`)** menggunakan dataset **Carabao Ripeness Level v19** di Google Colab dengan akselerasi GPU (T4).

> **PENTING SEBELUM MEMULAI:**  
> Pastikan runtime menggunakan GPU:  
> Klik menu **Runtime** > **Change runtime type** > Pilih **T4 GPU** > Klik **Save**.

### 1. Cek Ketersediaan GPU
Jalankan cell di bawah untuk memastikan NVIDIA GPU (T4) sudah aktif.

In [ ]:
!nvidia-smi

### 2. Instalasi Dependensi
Instal pustaka `ultralytics`, `roboflow`, dan `supervision`.

In [ ]:
!pip install -q ultralytics roboflow supervision

### 3. Unduh Dataset Beranotasi YOLOv11 dari Roboflow
Dataset akan otomatis diunduh dan diekstrak ke lingkungan Colab (582 train, 191 valid, 191 test).

In [ ]:
import os
from roboflow import Roboflow

# API Key Roboflow Anda
ROBOFLOW_API_KEY = "cQGSYleorD1JJbNwcS4r"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("mangosort-ai").project("carabao-ripeness-level-0x9me")
version = project.version(19)
dataset = version.download("yolov11")

print(f"\nDataset berhasil diunduh ke: {dataset.location}")

### 4. Melatih Model YOLO11 Nano di GPU
Kita akan melatih model dengan transfer learning dari `yolo11n.pt` sebanyak **50 epochs**.
Di GPU T4 Colab, 50 epochs hanya memakan waktu sekitar **5 - 10 menit**.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# 1. Load pretrained YOLO11 Nano
model = YOLO("yolo11n.pt")

# 2. Mulai proses training
yaml_path = f"{dataset.location}/data.yaml"

results = model.train(
    data=yaml_path,
    epochs=50,          # 50 epoch sangat ideal untuk konvergensi klasifikasi kematangan
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="mango",
    exist_ok=True,
    plots=True,
    verbose=True
)

# Simpan path direktori hasil training secara dinamis
TRAIN_DIR = Path(model.trainer.save_dir)
print(f"\n[SUKSES] Training selesai! File hasil disimpan di: {TRAIN_DIR}")

### 5. Evaluasi & Visualisasi Hasil Training
Melihat grafik loss, mAP, dan confusion matrix untuk memverifikasi akurasi kelas kematangan mangga.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# Cari folder hasil training
if 'TRAIN_DIR' in locals() and TRAIN_DIR.exists():
    save_dir = TRAIN_DIR
else:
    candidates = list(Path("runs").glob("**/weights/best.pt"))
    save_dir = candidates[0].parent.parent if candidates else Path("runs/detect/mango")

print(f"Memuat grafik dari direktori: {save_dir}\n")

# Tampilkan grafik pelatihan (Loss & mAP)
results_img = save_dir / "results.png"
if results_img.exists():
    print("Grafik Hasil Pelatihan (Loss dan mAP):")
    display(Image(filename=str(results_img)))
else:
    print(f"File {results_img} belum tersedia.")

# Tampilkan Confusion Matrix
cm_img = save_dir / "confusion_matrix.png"
if cm_img.exists():
    print("\nConfusion Matrix (Akurasi Prediksi per Kelas):")
    display(Image(filename=str(cm_img)))
else:
    print(f"File {cm_img} belum tersedia.")

### 6. Uji Coba Prediksi Gambar Test
Melihat langsung hasil deteksi pada beberapa gambar dari data uji.

In [ ]:
import glob
from IPython.display import Image, display

# Ambil 3 sampel gambar test
sample_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:3]

for img in sample_images:
    res = model.predict(source=img, conf=0.25, save=True, project="runs", name="test_pred", exist_ok=True)

# Tampilkan salah satu hasil prediksi
pred_images = glob.glob("runs/detect/test_pred/*.jpg") + glob.glob("runs/test_pred/*.jpg")
if pred_images:
    print("Contoh Hasil Deteksi:")
    display(Image(filename=pred_images[0]))

### 7. Unduh File `best.pt` ke Komputer Anda
Jalankan cell ini untuk mengunduh bobot `best.pt` yang sudah terlatih secara otomatis ke folder Download laptop Anda.

In [ ]:
from google.colab import files
from pathlib import Path

# Cari file best.pt secara otomatis
best_pt_candidates = list(Path("runs").glob("**/weights/best.pt"))

if best_pt_candidates:
    best_weights_path = best_pt_candidates[0]
    size_mb = best_weights_path.stat().st_size / (1024 * 1024)
    print(f"Ditemukan model terbaik: {best_weights_path} ({size_mb:.2f} MB)")
    print("Sedang mengunduh ke laptop Anda...")
    files.download(str(best_weights_path))
else:
    print("File best.pt belum ditemukan. Pastikan cell pelatihan (Cell 4) sudah selesai dieksekusi sampai epoch terakhir.")

---
### 📌 Langkah Terakhir di Laptop Anda:
Setelah `best.pt` selesai terunduh ke laptop:
1. Salin/pindahkan file `best.pt` tersebut ke folder proyek Anda di:
   `runs/detect/runs/mango/yolo11n_mango/weights/best.pt`  
   *(timpa/replace file lama)*.
2. Jalankan aplikasi web lokal Anda:
   ```powershell
   streamlit run app.py
   ```
Aplikasi web Anda akan langsung menggunakan model baru ini dengan akurasi deteksi dan klasifikasi kematangan yang jauh lebih tinggi!